# Final project, Option D: a cheaper PageIndex on the two books

Same corpus, same ten questions and same answering model (`gemini-2.5-flash`) as the Assignment 5 classic-RAG run. Only the retrieval method changes.

**How PageIndex retrieves:** (1) the book becomes a tree of its own headings; each node gets a short summary. (2) For each question the model reads the tree (titles and summaries, no vectors) and picks the nodes that should hold the answer. (3) The text of those nodes goes to the model, with the same answer prompt as A5.

**What is different from the first PageIndex run:** shorter node summaries (20 words, no lead-in), thinking off for the summaries, the tree placed before the question in the search prompt (so Gemini can cache it), and the search run twice: thinking off, then thinking on, to separate the two effects. Scoring is done by script against the locked ground truth; no hand ranking.

Run the cells top to bottom. Needs the Colab secret `GEMINI_API_KEY` (Notebook access ON). Upload the two corpus files when asked.

In [ ]:
# Step 1: install PageIndex. Its markdown parser builds the heading tree.
!pip install -q -U pageindex google-genai
import pageindex, importlib.metadata as md_
print("pageindex", md_.version("pageindex"))

In [ ]:
# Step 2: settings. Everything the comparison depends on lives here.
CHAT_MODEL = "gemini-2.5-flash"   # same as A5: used for summaries, tree search AND the answer
MAX_NODES  = 4                    # nodes the search may pick (A5 retrieved 4 chunks)
SUMMARY_WORDS = 20                # was 40; the first run overran it
SUMMARY_THINK = False             # was on
PRICE_IN, PRICE_CACHED, PRICE_OUT = 0.30, 0.03, 2.50   # USD per million tokens, Gemini 2.5 Flash paid tier
MAX_NODE_CHARS = 12000            # safety cap on one node's text sent to the answer prompt

import time, json, re, os
from google import genai
from google.genai import types
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
assert api_key, "No GEMINI_API_KEY secret (check Notebook access is ON)."
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(api_version="v1beta"))   # thinking controls need v1beta

def with_retry(fn, *a, **kw):
    for wait in [2, 5, 10, 20, 40, None]:
        try:
            return fn(*a, **kw)
        except Exception as e:
            if wait is None or "INVALID_ARGUMENT" in str(e) or "400" in str(e)[:20]: raise
            print(f"  retry in {wait}s ({str(e)[:80]})"); time.sleep(wait)

LAST = {"cached": 0}
def llm(prompt, think=True):
    """One Gemini call. Returns (text, prompt_tokens, answer_tokens, thought_tokens). think=False sets the thinking budget to 0."""
    cfg = None if think else types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_budget=0))
    r = with_retry(client.models.generate_content, model=CHAT_MODEL, contents=prompt, config=cfg)
    u = r.usage_metadata
    LAST["cached"] = getattr(u, "cached_content_token_count", 0) or 0
    return r.text, u.prompt_token_count or 0, u.candidates_token_count or 0, getattr(u, "thoughts_token_count", 0) or 0
print("Connected.", CHAT_MODEL)

In [ ]:
# Step 3: upload the two corpus files (Book1_..., Book2_...).
from google.colab import files
up = files.upload()
BOOKS = sorted(up)                      # Book1 sorts before Book2
for f in BOOKS:
    open(f, "wb").write(up[f])
    print(f, f"{len(up[f]):,} bytes")

In [ ]:
# Step 4: build the tree for each book with PageIndex's markdown parser (no LLM, no vectors).
from pageindex.page_index_md import md_to_tree

def flatten(nodes, path=()):
    out = []
    for n in nodes:
        p = path + (n["title"],)
        out.append({"node_id": n["node_id"], "title": n["title"], "path": " > ".join(p),
                    "text": n.get("text", ""), "line": n.get("line_num"), "depth": len(p) - 1})
        out += flatten(n.get("nodes", []), p)
    return out

NODES = []     # one flat list across both books
for bi, f in enumerate(BOOKS, start=1):
    tree = await md_to_tree(f, if_add_node_summary="no", if_add_node_text="yes",
                            if_add_node_id="yes", model=f"gemini/{CHAT_MODEL}")
    flat = flatten(tree["structure"])
    for n in flat:
        n["uid"] = f"B{bi}-{n['node_id']}"
        n["book"] = f"Book{bi}"
        n["path"] = f"Book{bi} > {n['path']}"
        n["chars"] = len(n["text"])
    NODES += flat
    print(f"{f}: {len(flat)} nodes, depth up to {max(n['depth'] for n in flat)}, "
          f"median {sorted(n['chars'] for n in flat)[len(flat)//2]} chars, largest {max(n['chars'] for n in flat):,}")
print("Total nodes:", len(NODES))

In [ ]:
# Step 5: one short summary per node. This is the ONLY indexing cost (the tree itself is free).
t0 = time.time()
tok_in = tok_out = tok_th = 0
for i, n in enumerate(NODES):
    body = n["text"].strip()
    if len(body) < 60:                       # heading-only node: nothing to summarize
        n["summary"] = ""
        continue
    p = (f"Summarize this section of a book in at most {SUMMARY_WORDS} words. Start directly with the content "
         f"(no lead-in such as 'This section'). Section title: {n['title']}\n\nTEXT:\n{body[:6000]}")
    s, a, b, c = llm(p, think=SUMMARY_THINK)
    n["summary"] = s.strip(); tok_in += a; tok_out += b; tok_th += c
    if i % 50 == 0: print(f"  {i}/{len(NODES)}")
INDEX = {"nodes": len(NODES), "seconds": round(time.time() - t0, 1),
         "prompt_tokens": tok_in, "answer_tokens": tok_out, "thought_tokens": tok_th}
print("Index built:", INDEX)
json.dump({"index": INDEX, "nodes": NODES}, open("pi_cheap_tree.json", "w"), indent=1)

In [ ]:
# Step 6: the tree the search model sees (titles, ids and summaries; NO node text).
def tree_view():
    lines = []
    for n in NODES:
        indent = "  " * n["depth"]
        s = f" :: {n['summary']}" if n["summary"] else ""
        lines.append(f"{indent}[{n['uid']}] {n['title']}{s}")
    return "\n".join(lines)
TREE = tree_view()
print(f"Tree view: {len(TREE):,} characters (about {len(TREE)//4:,} tokens). It is sent on every question.")
print(TREE[:1200])

In [ ]:
# Step 7: the PageIndex steps for one question.
BY_UID = {n["uid"]: n for n in NODES}

def search_prompt(q):
    # The tree comes first and is identical for every question, so Gemini's implicit cache can reuse it.
    return f"""You are given the tree index of two books (Book1 and Book2). Each line is a
section: [node id] title :: summary. Indentation shows nesting.
Your job is to pick the sections whose text most likely contains the answer to a question
(asked after the tree). Choose at most {MAX_NODES} node ids. If no section looks like it contains
the answer, return an empty list.

TREE:
{TREE}

QUESTION: {q}

Reply as JSON: {{"thinking": "<which sections and why>", "node_list": ["B1-0001", ...]}}"""

def answer_prompt(q, nodes):
    # Same instruction text as the A5 notebook; the labels name the node instead of a chunk number.
    ctx = "\n\n---\n\n".join(f"[Source: {n['uid']} | {n['path']}]\n{n['text'][:MAX_NODE_CHARS]}" for n in nodes)
    return f"""You are an assistant answering from supplied context only.

Using the context below, answer the question at the end. If the context does not
contain the answer, say you do not have enough information. Cite the chunk
numbers you used.

CONTEXT:
{ctx}

QUESTION: {q}
"""

def ask_pageindex(q, search_think=False):
    t0 = time.time()
    raw, s_in, s_out, s_th = llm(search_prompt(q), think=search_think)
    s_cached = LAST["cached"]
    t1 = time.time()
    m = re.search(r"\{.*\}", raw or "", re.S)
    try:
        j = json.loads(m.group(0)); picked, thinking = j.get("node_list", []), j.get("thinking", "")
    except Exception:
        picked, thinking = re.findall(r"B\d-\d{4}", raw or ""), raw
    picked = [u for u in dict.fromkeys(picked) if u in BY_UID][:MAX_NODES]
    nodes = [BY_UID[u] for u in picked]
    ans, a_in, a_out, a_th = llm(answer_prompt(q, nodes))
    t2 = time.time()
    return {"question": q, "answer": ans, "thinking": thinking, "picked": picked,
            "hits": [{"rank": i + 1, "node": n["uid"], "location": n["path"], "chars": n["chars"],
                      "text": n["text"][:MAX_NODE_CHARS]} for i, n in enumerate(nodes)],
            "search_s": round(t1 - t0, 2), "generation_s": round(t2 - t1, 2),
            "search_prompt_tokens": s_in, "search_cached_tokens": s_cached, "search_answer_tokens": s_out, "search_thought_tokens": s_th,
            "prompt_tokens": a_in, "answer_tokens": a_out, "thought_tokens": a_th,
            "context_chars": sum(min(n["chars"], MAX_NODE_CHARS) for n in nodes)}

r = ask_pageindex("What are the seven parts of a strong prompt?")
r = ask_pageindex("What are the seven parts of a strong prompt?")   # second call: look for cached tokens
print(r["picked"], r["search_s"], r["generation_s"], "cached tokens:", r["search_cached_tokens"]); print(r["answer"][:500])

In [ ]:
# Step 8: run the ten A5 questions twice (search thinking off, then on). Question wording identical to A5.
QUESTIONS = [
    ("Q1", "According to the research cited in AI's Missing Playbook, what share of organizations use AI in at least one function, and what share of enterprise AI projects produced no measurable effect on the bottom line?"),
    ("Q2", "What are the seven parts of a strong prompt?"),
    ("Q3", "In the three-step customer operations example, what is the Rolled Throughput Yield and the cost per verified chain completion?"),
    ("Q4", "How many evaluation trials are needed to confirm a 90% First Pass Yield at 95% confidence with a margin of error of plus or minus 5 points, and how many at plus or minus 2 points?"),
    ("Q5", "In the document review ledger, what did the organization budget per month versus actually spend, and what share of the true per-file cost was human review?"),
    ("Q6", "What does \"the machinist, not the machine\" mean as an AI architecture?"),
    ("Q7", "In AI's Missing Playbook, what is the third option besides build and buy, and what rule governs it?"),
    ("Q8", "What must the ninety-day proof point deliver, and which level of the AI Maturity Self-Assessment does achieving it correspond to?"),
    ("Q9", "What chunk size, in characters or tokens, does AI's Missing Playbook recommend for a RAG pipeline?"),
    ("Q10", "What are the three tiers of the Three-Tier Enterprise Data Taxonomy?"),
]
VARIANTS = {"cheap_nothink": False, "cheap_think": True}   # search-call thinking off / on (summaries are the same tree)
runs = {}
for vname, th in VARIANTS.items():
    res = []
    for qid, q in QUESTIONS:
        r = ask_pageindex(q, search_think=th); r["id"] = qid; res.append(r)
        print(f"{vname} {qid} | picked {r['picked']} | search {r['search_s']}s, {r['search_prompt_tokens']:,} in ({r['search_cached_tokens']:,} cached), {r['search_thought_tokens']} thought")
    runs[vname] = res
settings = dict(method="PageIndex-style tree search, cheaper settings (Option D)", chat_model=CHAT_MODEL, max_nodes=MAX_NODES,
                summary_words=SUMMARY_WORDS, summary_think=SUMMARY_THINK, max_node_chars=MAX_NODE_CHARS, n_nodes=len(NODES),
                files=BOOKS, pageindex_version=md_.version("pageindex"), run_at=time.strftime("%Y-%m-%d %H:%M"), tree_chars=len(TREE))
json.dump({"settings": settings, "index": INDEX, "runs": runs}, open("pi_cheap_results.json", "w"), indent=2)
print("Saved pi_cheap_results.json")

In [ ]:
# Step 9: automatic scoring (no hand ranking). Relevant passage = contains a required fact; answer correct = contains every required fact, or declines on Q9/Q10.
"""Automatic scoring against the locked A5 ground truth. No hand ranking.
- node/chunk is relevant if its text contains any one required fact (regex below)
- answer is correct if it contains every required fact group; Q9/Q10 are correct if it declines."""
import re
FACT = {  # any one match makes a retrieved passage relevant
 "Q1": r"88 percent|95 percent",
 "Q2": r"Success criteria",
 "Q3": r"83\.8|0\.239",
 "Q4": r"\b138\b|\b865\b",
 "Q5": r"25,300|\$105|89 ?%|89 percent",
 "Q6": r"deterministic engine|intelligent operator",
 "Q7": r"own what is durable|partner to learn",
 "Q8": r"measured win|Level 1",
}
ANS = {  # every group must match (case-insensitive)
 "Q1": [r"88", r"95"],
 "Q2": [r"persona", r"task", r"context", r"example", r"rules", r"success criteria", r"steps"],
 "Q3": [r"83\.8", r"0\.239"],
 "Q4": [r"138", r"865"],
 "Q5": [r"\$?105\b", r"25,300", r"89"],
 "Q6": [r"deterministic", r"operator|instructions"],
 "Q7": [r"partner", r"own what is durable|rent what is common|durable and specific"],
 "Q8": [r"measured win|before-and-after|before and after", r"level 1|proven"],
}
UNANSWERABLE = {"Q9", "Q10"}   # questions whose correct behavior is to decline; list your own ids here
DECLINE = r"do not have enough information|does not (?:specify|state|provide|name|list|give|recommend)|do not (?:specify|state)|not (?:specified|stated|named|listed)|no (?:specific )?(?:size|recommendation)|does not mention"
def relevant(qid, text):
    return qid in FACT and re.search(FACT[qid], text, re.I) is not None
def correct(qid, answer):
    if qid in UNANSWERABLE:
        return re.search(DECLINE, answer, re.I) is not None
    return all(re.search(g, answer, re.I) for g in ANS[qid])
def retrieval(qid, texts):
    """texts in rank order -> dict(recall, precision, first, rr)"""
    marks = [relevant(qid, t) for t in texts]
    first = next((i + 1 for i, m in enumerate(marks) if m), None)
    return dict(marks="".join("Y" if m else "N" for m in marks), recall=any(marks),
                precision=sum(marks) / len(marks) if marks else 0.0, first=first, rr=1 / first if first else 0.0)

qids = [q for q, _ in QUESTIONS]
def summarize(res):
    ans = [r for r in res if r["id"] in FACT]
    rets = {r["id"]: retrieval(r["id"], [h["text"] for h in r["hits"]]) for r in ans}
    mean = lambda xs: sum(xs) / len(xs)
    n = len(res)
    tin = mean([r["search_prompt_tokens"] + r["prompt_tokens"] for r in res]); cached = mean([r["search_cached_tokens"] for r in res])
    out = mean([r["search_answer_tokens"] + r["answer_tokens"] + r["search_thought_tokens"] + r["thought_tokens"] for r in res])
    th = mean([r["search_thought_tokens"] + r["thought_tokens"] for r in res])
    return dict(recall=f"{sum(v['recall'] for v in rets.values())}/{len(rets)}", precision=round(mean([v["precision"] for v in rets.values()]), 2),
                mrr=round(mean([v["rr"] for v in rets.values()]), 4), first_rank1=sum(1 for v in rets.values() if v["first"] == 1),
                answers_correct=f"{sum(correct(r['id'], r['answer']) for r in res)}/{n}",
                wrong=[r["id"] for r in res if not correct(r["id"], r["answer"])],
                mean_prompt_tokens=round(tin), mean_cached_tokens=round(cached), mean_thinking_tokens=round(th),
                mean_seconds=round(mean([r["search_s"] + r["generation_s"] for r in res]), 1),
                usd_per_1000_no_cache=round((tin * PRICE_IN + out * PRICE_OUT) / 1000, 2),
                usd_per_1000_with_cache=round(((tin - cached) * PRICE_IN + cached * PRICE_CACHED + out * PRICE_OUT) / 1000, 2),
                marks={q: v["marks"] for q, v in rets.items()})
summary = {v: summarize(r) for v, r in runs.items()}
for v, d in summary.items():
    print("\n==", v); [print(f"  {k}: {x}") for k, x in d.items()]
json.dump(summary, open("pi_cheap_summary.json", "w"), indent=2)
with open("pi_cheap_dump.md", "w") as f:
    for v, res in runs.items():
        f.write(f"\n# {v}\n")
        for r in res:
            f.write(f"\n## {r['id']}: {r['question']}\nPicked: {r['picked']} | correct: {correct(r['id'], r['answer'])}\n\nReasoning: {r['thinking']}\n\nAnswer: {r['answer']}\n")
for name in ("pi_cheap_results.json", "pi_cheap_summary.json", "pi_cheap_dump.md", "pi_cheap_tree.json"):
    files.download(name)
print("Saved and downloaded. Move the four pi_cheap_* files to FINAL PROJECT/RESULTS.")